In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
pip install statsforecast

In [ ]:
import numpy as np 
import pandas as pd
import matplotlib.pyplot as plt

# **Cinema Audience Forecasting challenge**
    To Predict daily theatre audience counts across multiple locations.

# ***Loading and Exploring Data***
Understanding the structure, quality, and patterns in our cinema attendance data.

In [ ]:
booknow_theaters = pd.read_csv('/kaggle/input/Cinema_Audience_Forecasting_challenge/booknow_theaters/booknow_theaters.csv')
cinepos_theaters = pd.read_csv('/kaggle/input/Cinema_Audience_Forecasting_challenge/cinePOS_theaters/cinePOS_theaters.csv')
theater_map = pd.read_csv('/kaggle/input/Cinema_Audience_Forecasting_challenge/movie_theater_id_relation/movie_theater_id_relation.csv')
cinepos_booking = pd.read_csv('/kaggle/input/Cinema_Audience_Forecasting_challenge/cinePOS_booking/cinePOS_booking.csv')
booknow_booking = pd.read_csv('/kaggle/input/Cinema_Audience_Forecasting_challenge/booknow_booking/booknow_booking.csv')
booknow_visits = pd.read_csv('/kaggle/input/Cinema_Audience_Forecasting_challenge/booknow_visits/booknow_visits.csv')
date_info = pd.read_csv('/kaggle/input/Cinema_Audience_Forecasting_challenge/date_info/date_info.csv')
print("Datasets loaded successfully!\n")
booknow_theaters
print("DATASET SHAPES")

print(f"CinePOS Theaters:     {cinepos_theaters.shape}")
print(f"BookNow Theaters:     {booknow_theaters.shape}")
print(f"Theater Mapping:      {theater_map.shape}")
print(f"CinePOS Booking:      {cinepos_booking.shape}")
print(f"BookNow Booking:      {booknow_booking.shape}") 
print(f"BookNow Visits:       {booknow_visits.shape}")
print(f"Date Info:            {date_info.shape}")

In [ ]:
print("DATASET OVERVIEW")

datasets = {
    'booknow_visits': booknow_visits,
    'booknow_theaters': booknow_theaters,
    'cinepos_theaters': cinepos_theaters,
    'theater_map': theater_map,
    'booknow_booking': booknow_booking,
    'cinepos_booking': cinepos_booking,
    'date_info': date_info
}

for name, df in datasets.items():
    print(f"\n{name}:")
    print(f"  Shape: {df.shape}")
    print(f"  Columns: {list(df.columns)}")
    missing = df.isnull().sum().sum()
    print(f"  Total missing values: {missing}")
    print(f"  View: {df.head(10)}")
    

In [ ]:
# coverage analysis
booknow_visits['show_date'] = pd.to_datetime(booknow_visits['show_date'])
date_info['show_date'] = pd.to_datetime(date_info['show_date'])

print("TEMPORAL COVERAGE")

print(f"\nBookNow Visits Data:")
print(f"  Date range: {booknow_visits['show_date'].min().date()} to {booknow_visits['show_date'].max().date()}")
print(f"  Total days: {(booknow_visits['show_date'].max() - booknow_visits['show_date'].min()).days + 1}")
print(f"  Total records: {len(booknow_visits):,}")
print(f"  Unique theaters: {booknow_visits['book_theater_id'].nunique()}")
print(f"  Average records per theater: {len(booknow_visits) / booknow_visits['book_theater_id'].nunique():.1f}")

print(f"\nDate Info (Calendar Data):")
print(f"  Date range: {date_info['show_date'].min().date()} to {date_info['show_date'].max().date()}")
print(f"  Total days covered: {len(date_info)}")
print(f"  Columns available: {list(date_info.columns)}")

## ***Data Overview***
- **Target Data**: 214,046 records across 826 theaters (Jan 2023 - Feb 2024)
- **No missing values** in target variable - excellent data quality!
- **Average 259 records per theater** - good temporal coverage(range of dates )

In [ ]:
print("TARGET VARIABLE: audience_count (from booknow_visits)")

print("\nStatistical Summary:")
print(booknow_visits['audience_count'].describe())

print(f"Missing values: {booknow_visits['audience_count'].isnull().sum()}")

# Visualize distribution
fig, axes = plt.subplots(1, 3, figsize=(18, 4))

# Histogram
axes[0].hist(booknow_visits['audience_count'], bins=50, edgecolor='black', alpha=0.7, color='steelblue')
axes[0].set_xlabel('Audience Count', fontsize=11)
axes[0].set_ylabel('Frequency', fontsize=11)
axes[0].set_title('Audience Count Distribution', fontsize=12, fontweight='bold')
axes[0].grid(True, alpha=0.3)

# Box plot
box_data = axes[1].boxplot(booknow_visits['audience_count'], vert=True, patch_artist=True)
box_data['boxes'][0].set_facecolor('lightcoral')
axes[1].set_ylabel('Audience Count', fontsize=11)
axes[1].set_title('Audience Count Boxplot', fontsize=12, fontweight='bold')
axes[1].grid(True, alpha=0.3, axis='y')

# Log scale histogram (to see distribution better)
axes[2].hist(booknow_visits[booknow_visits['audience_count'] > 0]['audience_count'], 
             bins=50, edgecolor='black', alpha=0.7, color='seagreen')
axes[2].set_xlabel('Audience Count (log scale)', fontsize=11)
axes[2].set_ylabel('Frequency', fontsize=11)
axes[2].set_title('Audience Count Distribution (excluding zeros)', fontsize=12, fontweight='bold')
axes[2].set_yscale('log')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## ***Target Variable Insights***
- **Mean**: 41.6 visitors/day, **Median**: 34 visitors/day (right-skewed)
- **Range**: 2 - 1,350 visitors 
- **High variability**: Std dev = 32.8 (79% of mean)
- **Outliers**: Many theaters exceed 400+ visitors (see boxplot)
- **Implication**: Need robust models that handle outliers and variability

## *Visualize Sample Theater Trends*


In [ ]:
# Visualize sample theater trends
sample_theater = booknow_visits['book_theater_id'].iloc[0]
theater_data = booknow_visits[booknow_visits['book_theater_id'] == sample_theater].sort_values('show_date')

fig, axes = plt.subplots(2, 1, figsize=(16, 8))

# Time series plot
axes[0].plot(theater_data['show_date'], theater_data['audience_count'], 
             marker='o', markersize=2, linewidth=1, color='darkblue', alpha=0.7)
axes[0].set_title(f'Audience Count Over Time - Theater: {sample_theater}', 
                  fontsize=14, fontweight='bold')
axes[0].set_xlabel('Date', fontsize=11)
axes[0].set_ylabel('Audience Count', fontsize=11)
axes[0].grid(True, alpha=0.3)

# Distribution for this theater
axes[1].hist(theater_data['audience_count'], bins=30, edgecolor='black', 
             alpha=0.7, color='coral')
axes[1].set_xlabel('Audience Count', fontsize=11)
axes[1].set_ylabel('Frequency', fontsize=11)
axes[1].set_title(f'Audience Distribution for Theater {sample_theater}', 
                  fontsize=14, fontweight='bold')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nTheater {sample_theater} Statistics:")
print(f"  Mean attendance: {theater_data['audience_count'].mean():.1f}")
print(f"  Median attendance: {theater_data['audience_count'].median():.1f}")
print(f"  Std deviation: {theater_data['audience_count'].std():.1f}")
print(f"  Min: {theater_data['audience_count'].min()}, Max: {theater_data['audience_count'].max()}")
print(f"  Zero attendance days: {(theater_data['audience_count'] == 0).sum()}")

In [ ]:
# Theater characteristics analysis
print("THEATER CHARACTERISTICS")

print("\nBookNow Theaters:")
print(f"  Total theaters: {len(booknow_theaters)}")
print(f"  Theater types: {booknow_theaters['theater_type'].nunique()}")
print("\n  Theater type distribution:")
for theater_type, count in booknow_theaters['theater_type'].value_counts().items():
    print(f"    {theater_type}: {count}")

print(f"\n  Theater areas: {booknow_theaters['theater_area'].nunique()}")
print("\n  Top 10 theater areas:")
for area, count in booknow_theaters['theater_area'].value_counts().head(10).items():
    print(f"    {area}: {count} theaters")

# Check for missing location data
print(f"\n  Missing latitude: {booknow_theaters['latitude'].isnull().sum()}")
print(f"  Missing longitude: {booknow_theaters['longitude'].isnull().sum()}")

print("\nCinePOS Theaters:")
print(f"  Total theaters: {len(cinepos_theaters)}")

print("\nTheater Mapping:")
print(f"  Total mappings: {len(theater_map)}")
print(f"  BookNow IDs mapped: {theater_map['book_theater_id'].nunique()}")
print(f"  CinePOS IDs mapped: {theater_map['cine_theater_id'].nunique()}")

## *Booking Data Analysis*


In [ ]:
# Booking data analysis
print("BOOKING DATA ANALYSIS")

print("\nBookNow Booking Data:")
print(f"  Total bookings: {len(booknow_booking):,}")
if 'show_datetime' in booknow_booking.columns:
    booknow_booking['show_datetime'] = pd.to_datetime(booknow_booking['show_datetime'])
    print(f"  Date range: {booknow_booking['show_datetime'].min()} to {booknow_booking['show_datetime'].max()}")
print(f"  Columns: {list(booknow_booking.columns)}")

print("\nCinePOS Booking Data:")
print(f"  Total bookings: {len(cinepos_booking):,}")
if 'show_datetime' in cinepos_booking.columns:
    cinepos_booking['show_datetime'] = pd.to_datetime(cinepos_booking['show_datetime'])
    print(f"  Date range: {cinepos_booking['show_datetime'].min()} to {cinepos_booking['show_datetime'].max()}")
print(f"  Columns: {list(cinepos_booking.columns)}")

# Check for tickets_sold in CinePOS
if 'tickets_sold' in cinepos_booking.columns:
    print(f"\n  Tickets sold statistics:")
    print(f"    Mean: {cinepos_booking['tickets_sold'].mean():.2f}")
    print(f"    Median: {cinepos_booking['tickets_sold'].median():.2f}")
    print(f"    Max: {cinepos_booking['tickets_sold'].max()}")
    print(f"    Total tickets: {cinepos_booking['tickets_sold'].sum():,}")

##  ***Theater Characteristics***
- **4 theater types**: Other (50%), Comedy (24%), Drama (22%), Action (5%)
- **103 geographic areas** - highly concentrated (top 10 areas = 382 theaters)
- **Missing coordinates**: 515 missing in BookNow, 7,722 in CinePOS
- **Theater mapping**: Only 150 BookNow theaters map to CinePOS (18%)

In [ ]:
# Date/Calendar information exploration
print("CALENDAR/DATE INFORMATION")

print(f"\nColumns in date_info: {list(date_info.columns)}")
print(f"Total unique dates: {date_info['show_date'].nunique()}")

# Display sample of date_info to understand what features are available
print(f"\nSample of date_info (first 15 rows):")
print(date_info.head(15).to_string())

# Check for any special indicators (holidays, weekends, etc.)
for col in date_info.columns:
    if col != 'show_date':
        unique_vals = date_info[col].nunique()
        print(f"\n  {col}: {unique_vals} unique values")
        if unique_vals <= 10:  # Show value counts if reasonable
            print(date_info[col].value_counts())

###  **Key Challenges Identified:**
1. **Missing location data** (62% of CinePOS theaters) - limits geospatial features
2. **Unbalanced theater types** - "Other" dominates
3. **High variance in attendance** - suggests theater-specific patterns
4. **Limited overlap** between BookNow & CinePOS (only 150 mapped)
5. **Future prediction** - no booking data for prediction period!

# *Base Line Model*

In [ ]:
df = booknow_visits.copy()
df.columns = ['unique_id', 'ds', 'y']
print(df.info())

***Unable to perform cross validation because some IDs have less than a week of data, let's drop them first.***

In [ ]:
threshold = 100
counts = df['unique_id'].value_counts()
sufficient_ids = counts[counts >= threshold].index

df = df[df['unique_id'].isin(sufficient_ids)]

In [ ]:
from statsforecast import StatsForecast
StatsForecast.plot(df)

In [ ]:
from statsforecast.models import (
    HoltWinters,
    CrostonClassic as Croston, 
    HistoricAverage,
    DynamicOptimizedTheta as DOT,
    SeasonalNaive,
    AutoARIMA
)

In [ ]:
# list of models and instantiation parameters
models = [
    HoltWinters(),
    Croston(),
    SeasonalNaive(season_length=7),
    HistoricAverage(),
    DOT(season_length=7),
    AutoARIMA(season_length=7)
]

In [ ]:
# Instantiate StatsForecast class as sf
sf = StatsForecast( 
    models=models,
    freq='D', 
    fallback_model = SeasonalNaive(season_length=7),
    n_jobs=-1,
)

In [ ]:
forecasts_df = sf.forecast(df=df, h=60 , level=[90])
forecasts_df.head()

In [ ]:
sf.plot(df,forecasts_df)

In [ ]:
# Plot to unique_ids and some selected models
sf.plot(df, forecasts_df, models=["HoltWinters","DynamicOptimizedTheta",'SeasonalNaive'], unique_ids=["book_00002", "book_00001"])

In [ ]:
sf.plot(df,forecasts_df, unique_ids=["book_00001"], models=["SeasonalNaive","HoltWinters"])

In [ ]:
cv_df = sf.cross_validation(
    df=df,
    h=7,
    step_size=7,
    n_windows=2
)

In [ ]:
cv_df.head()

In [ ]:
from utilsforecast.losses import mse

In [ ]:
def evaluate_cv(df, metric):
    models = df.columns.drop(['unique_id', 'ds', 'y', 'cutoff']).tolist()
    evals = metric(df, models=models)
    evals['best_model'] = evals[models].idxmin(axis=1)
    return evals

In [ ]:
evaluation_df = evaluate_cv(cv_df, mse)
evaluation_df.head()

In [ ]:
evaluation_df['best_model'].value_counts().to_frame().reset_index()

In [ ]:
# seasonal_ids = evaluation_df.query('best_model == "AutoARIMA"')['unique_id']
# sf.plot(df,forecasts_df, unique_ids=seasonal_ids, models=["AutoARIMA","HistoricAverage"])

In [ ]:
def get_best_model_forecast(forecasts_df, evaluation_df):
    with_best = forecasts_df.merge(evaluation_df[['unique_id', 'best_model']])
    res = with_best[['unique_id', 'ds']].copy()
    for suffix in ('', '-lo-90', '-hi-90'):
        res[f'best_model{suffix}'] = with_best.apply(lambda row: row[row['best_model'] + suffix], axis=1)
    return res

In [ ]:
prod_forecasts_df = get_best_model_forecast(forecasts_df, evaluation_df)
prod_forecasts_df.head()

In [ ]:
sf.plot(df, prod_forecasts_df, level=[90])

In [ ]:
sample = pd.read_csv('/kaggle/input/Cinema_Audience_Forecasting_challenge/sample_submission/sample_submission.csv')
forecasts = prod_forecasts_df.rename(columns={
    "unique_id": "book_theater_id",
    "ds": "show_date",
    "best_model": "audience_count"
})

forecasts["ID"] = forecasts["book_theater_id"] + "_" + forecasts["show_date"].astype(str)
sample["ID"] = sample["ID"].astype(str)

submission = sample.drop("audience_count", axis=1).merge(
    forecasts[["ID", "audience_count"]],
    how="left", on="ID"
)

submission["audience_count"] = submission["audience_count"].fillna(0)

submission[["ID", "audience_count"]].to_csv("submission.csv", index=False)

In [ ]:
forecasts.head()

In [ ]:
zero_count = (submission['audience_count'] == 0).sum()
print("Number of zero audience_count entries:", zero_count)

In [ ]:
submission.head

- ***BAELINE PUBLIC SCORE :0.176***
- ***With AutoArima : 0.166***

# ***Exploratory Data Analysis (EDA)***

Deep dive into temporal patterns, theater behaviors, and feature relationships.

## *Weekday vs Weekend Analysis*


In [ ]:
# Merge with date_info to get day_of_week
visits_with_day = booknow_visits.merge(date_info, on='show_date', how='left')

# Create weekend flag
visits_with_day['is_weekend'] = visits_with_day['day_of_week'].isin(['Saturday', 'Sunday'])

# Group by weekday
weekday_stats = visits_with_day.groupby('day_of_week')['audience_count'].agg(['mean', 'median', 'std', 'count'])
weekday_stats = weekday_stats.reindex(['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday'])

print("\nAudience Statistics by Day of Week:")
print(weekday_stats)

# Weekend vs Weekday comparison
weekend_comparison = visits_with_day.groupby('is_weekend')['audience_count'].agg(['mean', 'median', 'std', 'count'])
weekend_comparison.index = ['Weekday', 'Weekend']
print("\n\nWeekend vs Weekday Comparison:")
print(weekend_comparison)

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Bar plot by day of week
days_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
weekday_means = weekday_stats['mean'].reindex(days_order)
colors = ['steelblue']*5 + ['coral', 'coral']  # Highlight weekends
axes[0].bar(days_order, weekday_means, color=colors, alpha=0.7, edgecolor='black')
axes[0].set_xlabel('Day of Week', fontsize=12)
axes[0].set_ylabel('Average Audience Count', fontsize=12)
axes[0].set_title('Average Attendance by Day of Week', fontsize=14, fontweight='bold')
axes[0].tick_params(axis='x', rotation=45)
axes[0].grid(True, alpha=0.3, axis='y')

# Box plot comparison
weekend_data = [
    visits_with_day[~visits_with_day['is_weekend']]['audience_count'],
    visits_with_day[visits_with_day['is_weekend']]['audience_count']
]
bp = axes[1].boxplot(weekend_data, labels=['Weekday', 'Weekend'], patch_artist=True)
bp['boxes'][0].set_facecolor('steelblue')
bp['boxes'][1].set_facecolor('coral')
axes[1].set_ylabel('Audience Count', fontsize=12)
axes[1].set_title('Weekday vs Weekend Distribution', fontsize=14, fontweight='bold')
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

 Merged visits with date_info, created weekend flag, compared weekday vs weekend attendance by day of week.

**Results:**
- **26% Weekend Lift**: Average attendance increases from 38.5 (weekday) to 48.4 (weekend)
- **Monday Peak**: Highest weekday attendance at 47.6 (anomaly worth investigating)
- **Sunday Dominance**: Highest overall attendance at 51.7 average
- **Mid-Week Dip**: Tuesday/Wednesday lowest at ~35 average

**Inference:** Weekend timing is a **strong predictor**. Potential features: `is_weekend`, `is_monday`, `is_sunday`.


## *Temporal Trends Analysis*

In [ ]:
# Time Series Trends: Monthly and Overall Trends
print("TEMPORAL TRENDS ANALYSIS")

# Extract month and year
visits_with_day['year_month'] = visits_with_day['show_date'].dt.to_period('M')
visits_with_day['month'] = visits_with_day['show_date'].dt.month
visits_with_day['year'] = visits_with_day['show_date'].dt.year

# Monthly aggregation
monthly_avg = visits_with_day.groupby('year_month')['audience_count'].mean()

print(f"\nMonthly Average Attendance:")
print(monthly_avg.to_string())

# Detect trend
print(f"\n\nTrend Analysis:")
print(f"  First 3 months avg: {monthly_avg.iloc[:3].mean():.2f}")
print(f"  Last 3 months avg: {monthly_avg.iloc[-3:].mean():.2f}")
trend_pct = ((monthly_avg.iloc[-3:].mean() - monthly_avg.iloc[:3].mean()) / monthly_avg.iloc[:3].mean()) * 100
print(f"  Change: {trend_pct:+.1f}%")

# Visualize
fig, axes = plt.subplots(2, 1, figsize=(16, 10))

# Overall trend
daily_avg = visits_with_day.groupby('show_date')['audience_count'].mean().sort_index()
axes[0].plot(daily_avg.index, daily_avg.values, linewidth=1, alpha=0.6, color='steelblue')
# Add 7-day rolling average
daily_avg_rolling = daily_avg.rolling(window=7, center=True).mean()
axes[0].plot(daily_avg_rolling.index, daily_avg_rolling.values, linewidth=2.5, color='darkred', label='7-day MA')
axes[0].set_xlabel('Date', fontsize=12)
axes[0].set_ylabel('Average Audience Count', fontsize=12)
axes[0].set_title('Daily Average Attendance with 7-Day Moving Average', fontsize=14, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Monthly bars
monthly_avg_vals = [monthly_avg.iloc[i] for i in range(len(monthly_avg))]
month_labels = [str(monthly_avg.index[i]) for i in range(len(monthly_avg))]
axes[1].bar(month_labels, monthly_avg_vals, color='seagreen', alpha=0.7, edgecolor='black')
axes[1].set_xlabel('Month', fontsize=12)
axes[1].set_ylabel('Average Audience Count', fontsize=12)
axes[1].set_title('Monthly Average Attendance', fontsize=14, fontweight='bold')
axes[1].tick_params(axis='x', rotation=45)
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

Created year_month aggregation, calculated monthly averages, plotted daily trends with 7-day moving average.

**Results:**
- **Overall Trend**: -2.7% decline (first 3 months: 43.19 → last 3 months: 42.02)
- **Peak Months**: March 2023 (45.7) and December 2023 (46.3)
- **Seasonal Pattern**: Visible weekly oscillation in daily data (weekend spikes)

**Inference:** Weak declining trend but strong **weekly seasonality**. Use `month` and `day_of_week` features. Can Consider lag features to capture recent patterns.


## *Theater Type Analysis*

In [ ]:
# Theater Type Analysis
print("THEATER TYPE ANALYSIS")

# Merge with theater info
visits_with_theater = visits_with_day.merge(booknow_theaters[['book_theater_id', 'theater_type', 'theater_area']], 
                                             on='book_theater_id', how='left')

# Stats by theater type
type_stats = visits_with_theater.groupby('theater_type')['audience_count'].agg([
    'count', 'mean', 'median', 'std', 'min', 'max'
])
print("\nAudience Statistics by Theater Type:")
print(type_stats)

# Weekend effect by theater type
weekend_by_type = visits_with_theater.groupby(['theater_type', 'is_weekend'])['audience_count'].mean().unstack()
weekend_by_type.columns = ['Weekday', 'Weekend']
weekend_by_type['Weekend_Lift_%'] = ((weekend_by_type['Weekend'] - weekend_by_type['Weekday']) / weekend_by_type['Weekday'] * 100)
print("\n\nWeekend Effect by Theater Type:")
print(weekend_by_type)

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Average by type
type_means = type_stats['mean'].sort_values(ascending=False)
axes[0].barh(type_means.index, type_means.values, color='teal', alpha=0.7, edgecolor='black')
axes[0].set_xlabel('Average Audience Count', fontsize=12)
axes[0].set_ylabel('Theater Type', fontsize=12)
axes[0].set_title('Average Attendance by Theater Type', fontsize=14, fontweight='bold')
axes[0].grid(True, alpha=0.3, axis='x')

# Box plot by type
type_data = [visits_with_theater[visits_with_theater['theater_type'] == t]['audience_count'] 
             for t in type_stats.index]
bp = axes[1].boxplot(type_data, labels=type_stats.index, patch_artist=True)
for patch in bp['boxes']:
    patch.set_facecolor('lightblue')
axes[1].set_xlabel('Theater Type', fontsize=12)
axes[1].set_ylabel('Audience Count', fontsize=12)
axes[1].set_title('Attendance Distribution by Theater Type', fontsize=14, fontweight='bold')
axes[1].tick_params(axis='x', rotation=45)
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

Merged visits with theater info, calculated stats by theater_type, analyzed weekend effect variation.

**Results:**
- **Similar Averages**: Action=47.2, Comedy=47.0, Drama=43.6, Other=43.7
- **Weekend Lift Variation**: 
  - Action: 26.7%, Comedy: 23.4%
  - Drama: 30.4%, Other: 31.4%
- **Spread**: All types show high std (28-39), indicating within-type variability

**Inference:** Theater type alone is weak predictor, but **weekend interaction** varies. Can Consider theater-level encoding (mean/median) rather than just type.

## *Cinepos Booking Analysis*

In [ ]:
# CinePOS Booking Data 
print("CINEPOS BOOKING ANALYSIS")

# Aggregate CinePOS tickets by theater and date
cinepos_booking['show_date'] = pd.to_datetime(cinepos_booking['show_datetime']).dt.date
cinepos_daily = cinepos_booking.groupby(['cine_theater_id', 'show_date'])['tickets_sold'].sum().reset_index()
cinepos_daily['show_date'] = pd.to_datetime(cinepos_daily['show_date'])

# Map to BookNow theaters
cinepos_daily_mapped = cinepos_daily.merge(theater_map, on='cine_theater_id', how='inner')

# Aggregate by BookNow theater and date
booknow_cinepos_tickets = cinepos_daily_mapped.groupby(['book_theater_id', 'show_date'])['tickets_sold'].sum().reset_index()

# Merge with actual visits
correlation_data = visits_with_theater.merge(
    booknow_cinepos_tickets, 
    on=['book_theater_id', 'show_date'], 
    how='left'
)

# Check correlation
valid_correlation = correlation_data[correlation_data['tickets_sold'].notna()]
print(f"\nRecords with CinePOS ticket data: {len(valid_correlation):,} / {len(correlation_data):,} ({len(valid_correlation)/len(correlation_data)*100:.1f}%)")

if len(valid_correlation) > 0:
    corr = valid_correlation[['audience_count', 'tickets_sold']].corr().iloc[0, 1]
    print(f"Correlation between CinePOS tickets_sold and BookNow audience_count: {corr:.3f}")
    
    # Visualize
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    
    # Scatter plot
    sample_size = min(5000, len(valid_correlation))
    sample_data = valid_correlation.sample(n=sample_size, random_state=42)
    axes[0].scatter(sample_data['tickets_sold'], sample_data['audience_count'], 
                   alpha=0.3, s=10, color='darkblue')
    axes[0].set_xlabel('CinePOS Tickets Sold', fontsize=12)
    axes[0].set_ylabel('BookNow Audience Count', fontsize=12)
    axes[0].set_title(f'Correlation: {corr:.3f} (n={len(valid_correlation):,})', 
                     fontsize=14, fontweight='bold')
    axes[0].grid(True, alpha=0.3)
    
    # Side-by-side comparison
    avg_by_tickets = valid_correlation.groupby(pd.cut(valid_correlation['tickets_sold'], bins=10))['audience_count'].mean()
    axes[1].plot(range(len(avg_by_tickets)), avg_by_tickets.values, 
                marker='o', linewidth=2, markersize=8, color='darkred')
    axes[1].set_xlabel('Tickets Sold (binned)', fontsize=12)
    axes[1].set_ylabel('Average Audience Count', fontsize=12)
    axes[1].set_title('Audience Count by Ticket Sales Level', fontsize=14, fontweight='bold')
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
else:
    print("\n No overlapping data between CinePOS and BookNow for correlation analysis")

Aggregated CinePOS tickets by theater/date, mapped to BookNow IDs, merged with visits, calculated correlation.

**Results:**
- **Coverage**: Only 5.2% of records (11,150 out of 214,046)
- **Correlation**: Weak 0.267 between tickets_sold and audience_count
- **Data Quality**: Majority of prediction period lacks CinePOS data

**Inference:** **Skip CinePOS features** - insufficient coverage and weak predictive power don't justify complexity.[](http://)

# ***FEATURE ENGINEERING***

Based on above findings, prioritize:

✅ **Temporal Features:**
- `month`, `day_of_month`, `day_of_week` (from date)
- `is_weekend`, `is_monday`, `is_sunday` (binary flags)

✅ **Lag Features:**
- `audience_lag_1`, `audience_lag_7`, `audience_lag_14` (recent history)

✅ **Rolling Statistics:**
- `audience_roll_mean_7`, `audience_roll_mean_14` (trend capture)
- `audience_roll_std_7` (volatility capture)

❌ **Skip:**
- CinePOS features (5.2% coverage, 0.267 correlation)
- Theater type as categorical (similar means, high within-type variance)

In [ ]:
# Create comprehensive feature set
print("FEATURE ENGINEERING")

# Start with clean booknow_visits
df = booknow_visits.copy()
df['show_date'] = pd.to_datetime(df['show_date'])

# Merge with date_info for day_of_week
df = df.merge(date_info, on='show_date', how='left')

# Merge with theater characteristics
df = df.merge(booknow_theaters[['book_theater_id', 'theater_type', 'theater_area']], 
              on='book_theater_id', how='left')

print(f"\nBase dataset shape: {df.shape}")
print(f"Columns: {list(df.columns)}")

# 1. Temporal Features
df['month'] = df['show_date'].dt.month
df['day_of_month'] = df['show_date'].dt.day
df['dayofweek'] = df['show_date'].dt.dayofweek  # 0=Monday, 6=Sunday
df['is_weekend'] = df['day_of_week'].isin(['Saturday', 'Sunday']).astype(int)
df['is_monday'] = (df['day_of_week'] == 'Monday').astype(int)
df['is_sunday'] = (df['day_of_week'] == 'Sunday').astype(int)

# 2. Theater-level statistics (mean encoding)
theater_stats = df.groupby('book_theater_id')['audience_count'].agg([
    ('theater_mean', 'mean'),
    ('theater_median', 'median'),
    ('theater_std', 'std')
]).reset_index()
df = df.merge(theater_stats, on='book_theater_id', how='left')

# 3. Lag features (sort by theater and date first)
df = df.sort_values(['book_theater_id', 'show_date']).reset_index(drop=True)

# Previous day, week, 2 weeks
df['audience_lag_1'] = df.groupby('book_theater_id')['audience_count'].shift(1)
df['audience_lag_7'] = df.groupby('book_theater_id')['audience_count'].shift(7)
df['audience_lag_14'] = df.groupby('book_theater_id')['audience_count'].shift(14)

# 4. Rolling statistics (7-day and 14-day windows)
df['audience_roll_mean_7'] = df.groupby('book_theater_id')['audience_count'].transform(
    lambda x: x.rolling(window=7, min_periods=1).mean().shift(1)
)
df['audience_roll_std_7'] = df.groupby('book_theater_id')['audience_count'].transform(
    lambda x: x.rolling(window=7, min_periods=1).std().shift(1)
)
df['audience_roll_mean_14'] = df.groupby('book_theater_id')['audience_count'].transform(
    lambda x: x.rolling(window=14, min_periods=1).mean().shift(1)
)

print(f"\n Feature engineering complete!")
print(f"  Final dataset shape: {df.shape}")
print(f"  Total features: {len(df.columns) - 3}")  # Subtract target, date, theater_id

# Check for missing values
missing_summary = df.isnull().sum()
missing_summary = missing_summary[missing_summary > 0].sort_values(ascending=False)
print(f"\n  Missing values:")
if len(missing_summary) > 0:
    for col, count in missing_summary.items():
        pct = (count / len(df)) * 100
        print(f"    {col}: {count:,} ({pct:.1f}%)")
else:
    print("    None!")

# ***Model Traning***
## *Train/Test Split*

In [ ]:
# Prepare training data with time-based split
print("PREPARE TRAIN/TEST SPLIT")

# feature columns
feature_cols = [
    'month', 'day_of_month', 'dayofweek', 'is_weekend', 'is_monday', 'is_sunday',
    'theater_mean', 'theater_median', 'theater_std',
    'audience_lag_1', 'audience_lag_7', 'audience_lag_14',
    'audience_roll_mean_7', 'audience_roll_std_7', 'audience_roll_mean_14'
]

# categorical features
categorical_features = ['theater_type', 'theater_area', 'book_theater_id']

print(f"\nNumerical features ({len(feature_cols)}):")
for i, col in enumerate(feature_cols, 1):
    print(f"  {i}. {col}")

# Removeing rows with missing lag features (early dates per theater)
df_model = df.dropna(subset=feature_cols).copy()
print(f"\n Removed rows with missing lag features")
print(f"  Original: {len(df):,} rows")
print(f"  After dropna: {len(df_model):,} rows")
print(f"  Dropped: {len(df) - len(df_model):,} rows ({(len(df) - len(df_model))/len(df)*100:.1f}%)")

# Time-based split: 20% of dates for validation
unique_dates = sorted(df_model['show_date'].unique())
split_idx = int(len(unique_dates) * 0.8)
split_date = unique_dates[split_idx]

train_df = df_model[df_model['show_date'] < split_date].copy()
test_df = df_model[df_model['show_date'] >= split_date].copy()

print(f"\n Time-based split:")
print(f"  Split date: {split_date}")
print(f"  Train: {len(train_df):,} rows ({train_df['show_date'].min().date()} to {train_df['show_date'].max().date()})")
print(f"  Test:  {len(test_df):,} rows ({test_df['show_date'].min().date()} to {test_df['show_date'].max().date()})")

X_train = train_df[feature_cols]
y_train = train_df['audience_count']
X_test = test_df[feature_cols]
y_test = test_df['audience_count']

print(f"\nData prepared for modeling:")
print(f"  X_train shape: {X_train.shape}")
print(f"  X_test shape:  {X_test.shape}")

## *MODEL TRAINING & COMPARISON*

In [ ]:
# Train models and compare performance
print("MODEL TRAINING & COMPARISON")

from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

#  models to test
models = {
    'RandomForest (50)': RandomForestRegressor(n_estimators=50, max_depth=15, random_state=42, n_jobs=-1),
    'RandomForest (100)': RandomForestRegressor(n_estimators=100, max_depth=20, random_state=42, n_jobs=-1),
    'Ridge Regression': Ridge(alpha=1.0, random_state=42),
    'xgb_model': XGBRegressor(n_estimators=200,max_depth=8,learning_rate=0.1,subsample=0.8,random_state=4)
}

results = []

print("\nTraining models...")
for name, model in models.items():
    print(f"\n {name}")
    print(f"   Training...")
    model.fit(X_train, y_train)
    
    # Predictions
    train_pred = model.predict(X_train)
    test_pred = model.predict(X_test)
    
    # Metrics
    train_rmse = np.sqrt(mean_squared_error(y_train, train_pred))
    test_rmse = np.sqrt(mean_squared_error(y_test, test_pred))
    train_mae = mean_absolute_error(y_train, train_pred)
    test_mae = mean_absolute_error(y_test, test_pred)
    train_r2 = r2_score(y_train, train_pred)
    test_r2 = r2_score(y_test, test_pred)
    
    results.append({
        'Model': name,
        'Train RMSE': train_rmse,
        'Test RMSE': test_rmse,
        'Train MAE': train_mae,
        'Test MAE': test_mae,
        'Train R²': train_r2,
        'Test R²': test_r2
    })
    
    print(f"   Train RMSE: {train_rmse:.2f} | MAE: {train_mae:.2f} | R²: {train_r2:.3f}")
    print(f"   Test RMSE:  {test_rmse:.2f} | MAE: {test_mae:.2f} | R²: {test_r2:.3f}")

# Results summary
results_df = pd.DataFrame(results)
print("MODEL COMPARISON SUMMARY")
print(results_df.to_string(index=False))

# Select best model
best_model_name = results_df.loc[results_df['Test RMSE'].idxmin(), 'Model']
print(f"\n Best Model: {best_model_name}")
print(f"   Test RMSE: {results_df.loc[results_df['Test RMSE'].idxmin(), 'Test RMSE']:.2f}")
print(f"   Test MAE:  {results_df.loc[results_df['Test MAE'].idxmin(), 'Test MAE']:.2f}")

# Store best model
best_model = models[best_model_name]

## *FEATURE IMPORTANCE*

In [ ]:
# Feature importance analysis
print("FEATURE IMPORTANCE (Best Model)")

# feature importances from best RandomForest model
importances = best_model.feature_importances_
feature_importance_df = pd.DataFrame({
    'feature': feature_cols,
    'importance': importances
}).sort_values('importance', ascending=False)
    
print("\nTop 10 Most Important Features:")
print(feature_importance_df.head(10).to_string(index=False))
    
# Visualize
    
fig, ax = plt.subplots(figsize=(12, 6))
top_n = 15
top_features = feature_importance_df.head(top_n)
ax.barh(range(top_n), top_features['importance'].values, color='steelblue', alpha=0.7, edgecolor='black')
ax.set_yticks(range(top_n))
ax.set_yticklabels(top_features['feature'].values)
ax.set_xlabel('Importance', fontsize=12)
ax.set_title(f'Top {top_n} Feature Importances - {best_model_name}', fontsize=14, fontweight='bold')
ax.invert_yaxis()
ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()
    
print("\nKey Insights:")
print(f"  • Top feature: {feature_importance_df.iloc[0]['feature']} ({feature_importance_df.iloc[0]['importance']:.3f})")
print(f"  • Lag features: {feature_importance_df[feature_importance_df['feature'].str.contains('lag')]['importance'].sum():.3f}")
print(f"  • Rolling features: {feature_importance_df[feature_importance_df['feature'].str.contains('roll')]['importance'].sum():.3f}")
print(f"  • Theater stats: {feature_importance_df[feature_importance_df['feature'].str.contains('theater')]['importance'].sum():.3f}")

# ***SUBMISSION***

In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor
hgb_model = HistGradientBoostingRegressor(
    max_iter=150,              # Number of boosting iterations (like n_estimators)
    max_depth=5,               # Maximum tree depth
    learning_rate=0.05,         # Shrinkage rate
    max_leaf_nodes=31,         # Max leaves per tree (like num_leaves)
    min_samples_leaf=30,       # Minimum samples in one leaf
    l2_regularization=1, 
    early_stopping = False , # L2 regularization
    random_state=42
)

In [ ]:
# Retraining on full dataset for final predictions
print("RETRAIN ON FULL DATASET")

# Combine train and test for final model
X_full = df_model[feature_cols]
y_full = df_model['audience_count']

print(f"\nRetraining {best_model_name} on full dataset...")
print(f"  Training samples: {len(X_full):,}")

# final_model = RandomForestRegressor(n_estimators=50, max_depth=15, random_state=42, n_jobs=-1)
final_model = hgb_model
final_model.fit(X_full, y_full)

# Evaluate on full dataset
full_pred = final_model.predict(X_full)
full_rmse = np.sqrt(mean_squared_error(y_full, full_pred))
full_mae = mean_absolute_error(y_full, full_pred)
full_r2 = r2_score(y_full, full_pred)

print(f"\n Model retrained successfully!")
print(f"  Full dataset RMSE: {full_rmse:.2f}")
print(f"  Full dataset MAE:  {full_mae:.2f}")
print(f"  Full dataset R²:   {full_r2:.3f}")

# Save the feature columns and model info for later use
print(f"\n Model ready for predictions!")
print(f"  Features used: {len(feature_cols)}")
print(f"  Model type: {type(final_model).__name__}")

In [ ]:
sample_submission_path = '/kaggle/input/Cinema_Audience_Forecasting_challenge/sample_submission/sample_submission.csv'
official_sample = pd.read_csv(sample_submission_path)

print("LOADING SAMPLE SUBMISSION")
print(f"\n Official sample submission loaded!")
print(f"  Shape: {official_sample.shape}")
print(f"  Expected rows: 38,062")
print(f"  Actual rows: {len(official_sample):,}")
print(f"\nFirst few rows:")
print(official_sample.head())
print(f"\nLast few rows:")
print(official_sample.tail())

split_data = official_sample['ID'].str.split('_', expand=True)
official_sample['theater_id'] = split_data[1]
official_sample['date'] = pd.to_datetime(split_data[2])

print(f"\n Date range in sample submission:")
print(f"  Start: {official_sample['date'].min()}")
print(f"  End: {official_sample['date'].max()}")
print(f"  Days: {(official_sample['date'].max() - official_sample['date'].min()).days + 1}")
print(f"\n Unique theaters: {official_sample['theater_id'].nunique()}")

In [ ]:
# Prepare features for the exact theater-date combinations in sample submission
test_data = official_sample.copy()

# Map theater_id to book_theater_id format used in our training data
# The sample uses format like "00001" but our data uses "book_00001"
test_data['book_theater_id'] = 'book_' + test_data['theater_id']

# Extract temporal features
test_data['day_of_week'] = test_data['date'].dt.dayofweek
test_data['month'] = test_data['date'].dt.month
test_data['day'] = test_data['date'].dt.day
test_data['is_weekend'] = test_data['day_of_week'].isin([5, 6]).astype(int)
test_data['is_monday'] = (test_data['day_of_week'] == 0).astype(int)
test_data['is_sunday'] = (test_data['day_of_week'] == 6).astype(int)

# Merge with theater information
test_data = test_data.merge(
    booknow_theaters[['book_theater_id', 'theater_type', 'theater_area']], 
    on='book_theater_id', 
    how='left'
)

# Encode theater_type (use the same encoding as training)
if 'theater_type' in test_data.columns:
    test_data['theater_type_encoded'] = test_data['theater_type'].map({
        'single': 0, 'multiplex': 1, 'megaplex': 2
    }).fillna(1)  # Default to multiplex if unknown

print(f"\n Test data prepared!")
print(f"  Shape: {test_data.shape}")
print(f"  Missing values in key columns:")
print(test_data[['book_theater_id', 'date', 'theater_type']].isnull().sum())

In [ ]:
test_data.head()

In [ ]:
# Create lag features using the last known values from training data
print("\n Creating lag features from training data...")

# Get the last 14 days of data for each theater from training
last_date_train = booknow_visits['show_date'].max()
recent_train = booknow_visits[booknow_visits['show_date'] >= (last_date_train - pd.Timedelta(days=14))]

# Calculate statistics per theater
theater_recent_stats = recent_train.groupby('book_theater_id').agg({
    'audience_count': ['mean', 'std', 'min', 'max']
}).reset_index()
theater_recent_stats.columns = ['book_theater_id', 'lag_mean', 'lag_std', 'lag_min', 'lag_max']

# Merge with test data
test_data = test_data.merge(theater_recent_stats, on='book_theater_id', how='left')

# Fill missing lag features with overall means
for col in ['lag_mean', 'lag_std', 'lag_min', 'lag_max']:
    if col in test_data.columns:
        test_data[col] = test_data[col].fillna(test_data[col].mean())

print(f"  Lag features created!")
print(f"  Theaters with lag data: {test_data['lag_mean'].notna().sum()}")
print(f"  Theaters without lag data: {test_data['lag_mean'].isna().sum()}")

In [ ]:
# Check which features we need vs what we have
print("\n Feature Alignment Check:")
print(f"  Features needed for model: {len(feature_cols)}")
print(f"  Features in test_data: {len(test_data.columns)}")

# Show missing features
missing_features = [f for f in feature_cols if f not in test_data.columns]
print(f"\n Missing features ({len(missing_features)}):")
for f in missing_features:
    print(f"     - {f}")

# Show available features
available_features = [f for f in feature_cols if f in test_data.columns]
print(f"\n Available features ({len(available_features)}):")
for f in available_features[:10]:  # Show first 10
    print(f"     - {f}")
if len(available_features) > 10:
    print(f"     ... and {len(available_features)-10} more")

In [ ]:
# Creating all missing features
print("\n Creating missing features...")

# Add simple temporal features
test_data['day_of_month'] = test_data['date'].dt.day
test_data['dayofweek'] = test_data['date'].dt.dayofweek

# For lag and rolling features, we'll use the statistics we already calculated
# Map lag_mean to audience_lag_1, audience_lag_7, audience_lag_14
test_data['audience_lag_1'] = test_data['lag_mean']
test_data['audience_lag_7'] = test_data['lag_mean'] 
test_data['audience_lag_14'] = test_data['lag_mean']

# Rolling means and std
test_data['audience_roll_mean_7'] = test_data['lag_mean']
test_data['audience_roll_mean_14'] = test_data['lag_mean']
test_data['audience_roll_std_7'] = test_data['lag_std']

# Theater aggregations (mean, median, std per theater)
test_data['theater_mean'] = test_data['lag_mean']
test_data['theater_median'] = test_data['lag_mean']  # Approximation
test_data['theater_std'] = test_data['lag_std']

print("All features created!")

In [ ]:
print("\n Preparing features for prediction...")

X_test_submission = test_data[feature_cols].copy()

# Fill any remaining missing values
X_test_submission = X_test_submission.fillna(X_test_submission.mean())

print(f"  Features prepared!")
print(f"  Shape: {X_test_submission.shape}")
print(f"  Feature columns: {len(feature_cols)}")
print(f"  Missing values: {X_test_submission.isnull().sum().sum()}")

# Make predictions using the trained model
print(f"\n Making predictions with {type(final_model).__name__}...")
test_predictions = final_model.predict(X_test_submission)

# Ensure no negative predictions (minimum audience count should be 0)
test_predictions = np.maximum(test_predictions, 0)

print(f"  Predictions complete!")
print(f"  Prediction statistics:")
print(f"    Mean: {test_predictions.mean():.2f}")
print(f"    Median: {np.median(test_predictions):.2f}")
print(f"    Min: {test_predictions.min():.2f}")
print(f"    Max: {test_predictions.max():.2f}")
print(f"    Std: {test_predictions.std():.2f}")

In [ ]:
print("CREATING FINAL SUBMISSION FILE")

# submission DataFrame
final_submission = official_sample[['ID']].copy()
final_submission['audience_count'] = test_predictions

final_submission['audience_count'] = final_submission['audience_count'].round().astype(int)


print(f"\n Submission Statistics:")
print(f"  Total predictions: {len(final_submission):,}")
print(f"  Missing values: {final_submission['audience_count'].isnull().sum()}")
print(f"  Negative values: {(final_submission['audience_count'] < 0).sum()}")
print(f"  Zero values: {(final_submission['audience_count'] == 0).sum()}")
print(f"  Mean prediction: {final_submission['audience_count'].mean():.2f}")
print(f"  Median prediction: {final_submission['audience_count'].median():.2f}")

final_submission.to_csv('/kaggle/working/submission.csv', index=False)

In [ ]:
final_submission.head()